# Results Math Explorer

This notebook loads `results_full_v2.csv` and calculates statistics using filters.
It is intended for Stage 1 NORLR results, including Arabica-to-Robusta and Robusta-to-Arabica evaluations.

Each run has two evaluation rows:
- same-variety evaluation: trained and tested on the same variety
- cross-variety evaluation: trained on one variety and tested on the other

In [2]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda value: f'{value:.4f}')

CSV_PATH = Path('results_full_v2.csv')
df = pd.read_csv(CSV_PATH)

print(f'Loaded {len(df):,} evaluation rows')
print(f'Runs: {df["folder_name"].nunique():,}')
print(df[['folder_name', 'train_variety', 'model', 'optimizer', 'test_variety']].head())

Loaded 120 evaluation rows
Runs: 60
                                         folder_name train_variety  \
0  fixed_NORLR_arabica_efficientnet_adam_lr1e-4_b...       arabica   
1  fixed_NORLR_arabica_efficientnet_adam_lr1e-4_b...       arabica   
2  fixed_NORLR_arabica_efficientnet_adamw_lr1e-4_...       arabica   
3  fixed_NORLR_arabica_efficientnet_adamw_lr1e-4_...       arabica   
4  fixed_NORLR_arabica_efficientnet_adan_lr1e-4_b...       arabica   

          model optimizer test_variety  
0  efficientnet      adam         same  
1  efficientnet      adam        other  
2  efficientnet     adamw         same  
3  efficientnet     adamw        other  
4  efficientnet      adan         same  


## Filter Helpers

Use `filter_results` to select training variety, test variety, model, optimizer, learning rate, batch size, or dropout.
For cross-variety filtering, use `train_variety` and `tested_variety` together.

In [4]:
def filter_results(data, train_variety=None, tested_variety=None, model=None, optimizer=None,
                   lr=None, batch_size=None, dropout=None, test_type=None):
    result = data.copy()

    if train_variety is not None:
        result = result[result['train_variety'].eq(train_variety)]
    if tested_variety is not None:
        result = result[result['eval_label'].str.contains(f'tested:{tested_variety}', regex=False)]
    if model is not None:
        result = result[result['model'].eq(model)]
    if optimizer is not None:
        result = result[result['optimizer'].eq(optimizer)]
    if batch_size is not None:
        result = result[result['batch_size'].eq(batch_size)]
    if dropout is not None:
        result = result[result['dropout'].eq(dropout)]
    if test_type is not None:
        result = result[result['test_variety'].eq(test_type)]

    return result.reset_index(drop=True)


def cross_variety(data, train_variety, tested_variety):
    if train_variety == tested_variety:
        raise ValueError('Cross-variety analysis requires different training and testing varieties.')
    return filter_results(data, train_variety=train_variety, tested_variety=tested_variety)


def describe_metric(data, metric='macro_avg__f1-score'):
    if metric not in data.columns:
        raise ValueError(f'Unknown metric: {metric}')
    values = pd.to_numeric(data[metric], errors='coerce').dropna()
    if values.empty:
        return pd.DataFrame(columns=['count', 'mean', 'median', 'std', 'min', 'max'])
    return pd.DataFrame({

_IncompleteInputError: incomplete input (673330556.py, line 35)